# hnc on Colab

Run the full Mapillary -> TRIBE v2 -> GeoParquet 2.0 pipeline on a Colab Pro **L4 24GB** runtime.

**Before you start**

1. **Runtime > Change runtime type > L4 GPU**.
2. Open the **Secrets** tab (key icon, left sidebar) and add:
   - `MAPILLARY_ACCESS_TOKEN` (from https://www.mapillary.com/dashboard/developers)
   - `HF_TOKEN` (from https://huggingface.co/settings/tokens, with read access, and accept the [Llama 3.2 license](https://huggingface.co/meta-llama/Llama-3.2-3B) on Hugging Face first)
3. Run all cells top to bottom.

Repo, https://github.com/walkthru-earth/hnc

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf hnc
!git clone --depth 1 https://github.com/walkthru-earth/hnc.git
%cd /content/hnc

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!uv --version

In [ ]:
from google.colab import userdata
import os

os.environ['MAPILLARY_ACCESS_TOKEN'] = userdata.get('MAPILLARY_ACCESS_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Colab pre-sets MPLBACKEND='module://matplotlib_inline.backend_inline'
# which leaks into uv subprocesses and breaks mne -> matplotlib import
# during TRIBE v2 load. Force a headless backend.
os.environ['MPLBACKEND'] = 'Agg'

with open('.env', 'w') as f:
    f.write(f"MAPILLARY_ACCESS_TOKEN={os.environ['MAPILLARY_ACCESS_TOKEN']}\n")
    f.write(f"HF_TOKEN={os.environ['HF_TOKEN']}\n")
    f.write("MPLBACKEND=Agg\n")
print('secrets loaded, .env written, MPLBACKEND forced to Agg')

In [ ]:
!uv sync --extra tribe 2>&1 | tail -20

In [ ]:
# Override torch with Blackwell-compatible cu128 wheels in the project venv.
# Without -p .venv/bin/python, `uv pip install` lands in the system /usr env
# and `uv run` keeps using the original torch from .venv (sm_50..sm_90 only).
# Colab Pro can hand you an RTX PRO 6000 Blackwell (sm_120). cu128 wheels
# include sm_120 on top of all older capabilities, so this is safe on Hopper,
# Ada, and Ampere too.
!uv pip install -p /content/hnc/.venv/bin/python --reinstall \
  --index-url https://download.pytorch.org/whl/cu128 \
  torch torchvision 2>&1 | tail -10
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'arch_list', torch.cuda.get_arch_list())"

In [ ]:
!uv run python -c "import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
!MPLBACKEND=Agg uv run hnc-run \
  --bbox-name borough \
  --max-images 50 \
  --start-captured-at 2020-01-01 \
  --device cuda \
  --out /content/hnc_borough.parquet

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
df = con.execute("SELECT image_id, captured_at, lon, lat, compass_angle, len(brain_activity) AS n_vertices FROM read_parquet('/content/hnc_borough.parquet') LIMIT 10").fetchdf()
df

In [ ]:
from google.colab import files
files.download('/content/hnc_borough.parquet')